In [ ]:
import re
import os
import json

from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

import matplotlib.pyplot as plt

1. ??????????? ??????? ???? ??????? ?????? ???????????? ???????????? ?????? ??? ????? ?????? ? ??? ???????? ???????? ?????? ("model_bounding_sphere_strict_outer" ? "hull_bounding_sphere_strict_outer").


2. ??????????? ??????? ???? ??????? ?????? ???????????? ????????????, ????????????? ???????????? ?????? ????? ????, ?????? ??? ????? ?????? ? ??? ???????? ???????? ?????? ("model_bounding_sphere_strict_outer_absolute" ? "hull_bounding_sphere_strict_outer_absolute").


3. ??????????? ??????? ???? ??????? ?????? ???????????? ???????????? ?????? ??? ????? ?????? ? ??? ???????? ???????? ?????? ("model_bounding_sphere_missed" ? "hull_bounding_sphere_missed").


4. ??????????? ????????????? ??????????? ???? ?? ??????????????? ?????? ("model_bounding_sphere_concentric_sphere" ? "hull_bounding_sphere_concentric_sphere")

In [ ]:
models = pd.read_csv('data/models.csv')
models.head()

In [ ]:
def histogram_exist(model_name):
    return os.path.isfile(f"data/histograms/{model_name}.json")
    
def torus_clean(model_type):
    return re.sub(r'(Torus).*', '\g<1>', model_type)

In [ ]:
# ??????? ?????? ?? ??????, ??? ??????? ????????? ???????????
# ? ???????? ????
models = models[models.name.apply(histogram_exist)]
models.type = models.type.apply(torus_clean)
models.reset_index(drop=True, inplace=True)
models.type.value_counts()

In [ ]:
def get_histogram(model_name, intervals=32):
    with open(f"data/histograms/{model_name}.json", "r") as file:
        data = json.load(file)
    res_histogram = dict()
    for histogram in data['histogram_data']:
        if histogram['intervals'] == intervals:
            res_histogram[histogram['type']] = histogram['data']
    return res_histogram

In [ ]:
histograms = pd.DataFrame(list(models.name.apply(get_histogram)))

In [ ]:
models_data = pd.concat([models, histograms], axis=1)

label_encoder = LabelEncoder()
models_data.type = label_encoder.fit_transform(models_data.type)
models_data.head()

In [ ]:
# ????????? ??????? ????? ??? ???????????
X = models_data.drop(['name', 'type'], axis=1)
X = np.apply_along_axis(np.hstack, 1, X)
y = models_data.type.values

In [ ]:
# 4-5 ???????? ???????? ??? ??????????? ?????, ??? ???????
K = range(1, 10)
clfs = [KMeans(n_clusters=k, random_state=42).fit(X) for k in K]
dist = [clf.inertia_ for clf in clfs]

plt.plot(K, dist, marker='o')
plt.xlabel('k')
plt.ylabel('Sum of distances')
plt.title('The Elbow Method showing the optimal k')
plt.show()

In [ ]:
def get_accuracy(X, y_true):
    clf = KMeans(n_clusters=5, random_state=42)
    clf.fit(X)
    y_pred = np.zeros_like(y_true)
    for label in np.unique(clf.labels_):
        mask = clf.labels_ == label
        y_pred[mask] = Counter(y_true[mask]).most_common(1)[0][0]
    
    return accuracy_score(y_true, y_pred)

In [ ]:
get_accuracy(X, y)

In [ ]:
y = models_data.type.values
res_accuracy = 0
res_labels = []
for number_of_labels in range(2, 9):
    for labels in list(combinations(models_data.keys()[2:], number_of_labels)):
        labels = list(labels)
        X = models_data[labels]
        X = np.apply_along_axis(np.hstack, 1, X)
        cur_accuracy = get_accuracy(X, y)
        if cur_accuracy > res_accuracy:
            res_accuracy = cur_accuracy
            res_labels = labels

In [ ]:
# ????????? ?????, ?? ?? ????????? ??? ????????? ????
# ????? ????? ???? ???????????
print(res_labels, res_accuracy)